# TorlakTag: Multi-run (xlm-roberta-large, mt5-base, mdeberta-v3-base) + EXB export (TOR_C_0031)

This notebook trains 3 transformer backbones with 4 token-level heads (LEMMA/UPOS/FEATS/XPOS),
keeps **best_model.pt** based on dev `full_acc`, and exports predictions on **TOR_C_0031.exb**
to **CoNLL-U** with sentence-level metadata:

- sent_id, text
- speaker (ID), speaker_abbr
- location
- speaker_age, speaker_gender, speaker_education (education defaults to `no education`)
- start_time, end_time

> Set the two paths in the **CONFIG** cell (`DATA_ROOT` and `MTE2UD_PATH`) if your Drive layout differs.

In [ ]:
# =========================
# 0) SETUP
# =========================
!pip -q install lxml

import os, re, json, math, time, random
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from collections import Counter

import pandas as pd
from lxml import etree

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

from google.colab import drive
drive.mount("/content/drive")

print("🧠 torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

Mounted at /content/drive
🧠 torch: 2.9.0+cu128 | cuda: True


In [ ]:
# =========================
# 1) CONFIG
# =========================

DATA_ROOT   = Path("/content/drive/MyDrive/TorlakData")               # contains tor_train/tor_dev/tor_test, speaker meta, exb files
MTE2UD_PATH = DATA_ROOT / "mte2ud_output.txt"                        # mapping file

# Speaker metadata file (TSV). Must contain NAME column (speaker abbrev) and ideally ID/LOCATION/AGE/GENDER/EDUCATION.
SPK_META_PATH = DATA_ROOT / "spk.metadata.txt"

# EXB folder root (where TOR_C_0031.exb lives). If unknown, we will auto-find under /content/drive/MyDrive.
EXB_DIR = DATA_ROOT / "exb_corrected"

# Optional: place -> lat/lon mapping table: columns place, lat, lon (tab-separated)
GEO_TSV = None  # e.g. DATA_ROOT / "place_geo.tsv"

# Training outputs
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
MODELS_ROOT = Path(f"/content/drive/MyDrive/TorlakTag/models/multirun_{RUN_TAG}")
PRED_ROOT   = Path(f"/content/drive/MyDrive/TorlakData/conllu_preds_{RUN_TAG}")
MODELS_ROOT.mkdir(parents=True, exist_ok=True)
PRED_ROOT.mkdir(parents=True, exist_ok=True)

# Training defaults
EPOCHS   = 30
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
GRAD_CLIP    = 1.0
PATIENCE     = 4
FREEZE_EPOCHS = 1

SEED = 13
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("📁 MODELS_ROOT:", MODELS_ROOT)
print("📁 PRED_ROOT  :", PRED_ROOT)
print("🧠 device:", device)

📁 MODELS_ROOT: /content/drive/MyDrive/TorlakTag/models/multirun_20260213_064842
📁 PRED_ROOT  : /content/drive/MyDrive/TorlakData/conllu_preds_20260213_064842
🧠 device: cuda


In [ ]:
# =========================
# 2) TOKEN NORMALIZATION (from your old notebook)
# =========================

def apply_spec_mapping(s: str) -> str:
    if s is None:
        return ""
    s = s.replace("#", "")
    s = s.replace("W", "Ə").replace("w", "ə")
    s = s.replace("1", "ḱ").replace("6", "ḱ")
    s = s.replace("2", "ǵ")
    s = s.replace("3", "č")
    s = s.replace("x", "š").replace("X", "š")
    s = s.replace("5", "ƨ")
    s = s.replace("ššš", "XXX")
    return s

RE_OVERLAP   = re.compile(r"\[[^\]]*\]")
RE_DOUBLEPAR = re.compile(r"^\(\(.*\)\)$")
RE_BULLETS   = re.compile(r"^[•]+$")
RE_LONGVOWEL = re.compile(r"([aeiouə])\1+")
RE_SPACES    = re.compile(r"\s+")

_STRIP_EDGE = " \t\r\n\"'“”„`´.,;:!?(){}[]<>•"

def is_x_special_token(tok: str) -> bool:
    t = tok.strip()
    if not t:
        return True
    if RE_DOUBLEPAR.match(t):
        return True
    if RE_BULLETS.match(t):
        return True
    return False

def strip_attached_specials(tok: str) -> str:
    t = tok.strip()
    t = re.sub(r"/+$", "", t)      # rek/ -> rek
    t = t.strip(_STRIP_EDGE)
    return t

def normalize_word(tok: str) -> str:
    t = apply_spec_mapping(tok).lower()
    t = RE_LONGVOWEL.sub(r"\1", t)
    t = RE_SPACES.sub(" ", t).strip()
    return t

def tokenize_with_rules(raw_text: str):
    if raw_text is None:
        return [], []
    s = apply_spec_mapping(raw_text).lower()
    s = RE_OVERLAP.sub(" ", s)

    raw_tokens = [t for t in s.split() if t.strip()]
    tokens, special = [], []
    for rt in raw_tokens:
        if is_x_special_token(rt):
            tokens.append(rt)
            special.append(True)
            continue

        has_alnum = any(ch.isalpha() or ch.isdigit() for ch in rt)
        if has_alnum:
            w = strip_attached_specials(rt)
            w = normalize_word(w)
            if w:
                tokens.append(w)
                special.append(False)
            else:
                tokens.append(rt)
                special.append(True)
        else:
            tokens.append(rt)
            special.append(True)

    return tokens, special

print(tokenize_with_rules("((?)) stoju/ rek/ •• aaa əəə [overlap] test"))

(['((?))', 'stoju', 'rek', '••', 'a', 'ə', 'test'], [True, False, False, True, False, False, False])


In [ ]:
# =========================
# 3) LOAD MTE→UD MAPPING + SPLITS
# =========================

UPOS_SET = {
    "ADJ","ADP","ADV","AUX","CCONJ","DET","INTJ","NOUN","NUM","PART",
    "PRON","PROPN","PUNCT","SCONJ","SYM","VERB","X"
}

def load_mte2ud(path: Path):
    m = {}
    bad = 0
    with path.open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            parts = re.split(r"\t+", line)
            if len(parts) < 3:
                parts = re.split(r"\s{2,}", line)
            if len(parts) < 3:
                bad += 1
                continue

            mte = parts[0].strip()
            upos = parts[1].strip()

            if len(parts) >= 4 and parts[2].strip() in UPOS_SET and upos in UPOS_SET:
                feats = parts[3].strip()
            else:
                feats = parts[2].strip()

            feats = feats if feats else "_"
            m[mte] = (upos if upos else "X", feats)

    print(f"✅ loaded MTE→UD mapping: {len(m)} tags (skipped {bad} broken lines)")
    return m

mte2ud = load_mte2ud(MTE2UD_PATH)

def find_split_file(root: Path, stem: str) -> Path:
    for ext in ["", ".txt", ".tsv", ".conllu", ".conll", ".data"]:
        p = root / f"{stem}{ext}"
        if p.exists():
            return p
    hits = [h for h in root.rglob(f"{stem}*") if h.is_file()]
    if hits:
        hits = sorted(hits, key=lambda x: len(str(x)))
        return hits[0]
    raise FileNotFoundError(f"Could not find split file for '{stem}' under {root}")

TRAIN_PATH = find_split_file(DATA_ROOT, "tor_train")
DEV_PATH   = find_split_file(DATA_ROOT, "tor_dev")
TEST_PATH  = find_split_file(DATA_ROOT, "tor_test")

print("📄 train:", TRAIN_PATH)
print("📄 dev  :", DEV_PATH)
print("📄 test :", TEST_PATH)

def read_tok_lemma_mte(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if len(parts) < 3:
                parts = line.split()
            if len(parts) < 3:
                continue
            form, lemma, mte = parts[0], parts[1], parts[2]

            # normalize similarly to EXB processing
            tks, sp = tokenize_with_rules(form)
            if not tks:
                continue
            form_n = tks[0]
            is_special = sp[0]

            if is_special:
                rows.append((form_n, form_n, "X"))  # forced X tag
                continue

            lemma_n = normalize_word(strip_attached_specials(lemma))
            if not lemma_n:
                lemma_n = form_n

            rows.append((form_n, lemma_n, mte))
    return rows

train_raw = read_tok_lemma_mte(TRAIN_PATH)
dev_raw   = read_tok_lemma_mte(DEV_PATH)
test_raw  = read_tok_lemma_mte(TEST_PATH)

print(f"✅ loaded splits: train={len(train_raw)} dev={len(dev_raw)} test={len(test_raw)}")
print("sample:", train_raw[:5])

def to_ud_example(row):
    form, lemma, xpos = row
    if xpos == "X":
        return (form, lemma, "X", "_", "X")
    upos, feats = mte2ud.get(xpos, ("X", "_"))
    return (form, lemma, upos, feats, xpos)

train_ex = [to_ud_example(r) for r in train_raw]
dev_ex   = [to_ud_example(r) for r in dev_raw]
test_ex  = [to_ud_example(r) for r in test_raw]

print("Converted sample:", train_ex[:5])

✅ loaded MTE→UD mapping: 1198 tags (skipped 0 broken lines)
📄 train: /content/drive/MyDrive/TorlakData/tor_train.tsv
📄 dev  : /content/drive/MyDrive/TorlakData/tor_dev.tsv
📄 test : /content/drive/MyDrive/TorlakData/tor_test.tsv
✅ loaded splits: train=52349 dev=6486 test=6424
sample: [('pa', 'pa', 'Cc'), ('sam', 'biti', 'Var1s'), ('mu', 'on', 'Pp3msd'), ('pričala', 'pričati', 'Vmp-sf'), ('lazaričke', 'lazarički', 'Agpfpa')]
Converted sample: [('pa', 'pa', 'CCONJ', '_', 'Cc'), ('sam', 'biti', 'AUX', 'Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbForm=Fin', 'Var1s'), ('mu', 'on', 'PRON', 'Case=Dat|Gender=Masc|Number=Sing|Person=3|PronType=Prs', 'Pp3msd'), ('pričala', 'pričati', 'VERB', 'Gender=Fem|Number=Sing|Tense=Past|VerbForm=Part|Voice=Act', 'Vmp-sf'), ('lazaričke', 'lazarički', 'ADJ', 'Case=Acc|Degree=Pos|Gender=Fem|Number=Plur', 'Agpfpa')]


In [ ]:
# =========================
# 4) SPEAKER METADATA (add age, gender, education)
# =========================

df_spk = pd.read_csv(SPK_META_PATH, sep="\t", dtype=str, keep_default_na=False)
df_spk.columns = [c.strip() for c in df_spk.columns]
for c in df_spk.columns:
    df_spk[c] = df_spk[c].astype(str).str.strip()

def _pick_col(cols, keywords):
    cols_l = {c.lower(): c for c in cols}
    for kw in keywords:
        for c in cols:
            if kw in c.lower():
                return c
    return None

COL_NAME = _pick_col(df_spk.columns, ["name"])
COL_ID   = _pick_col(df_spk.columns, ["id"])
COL_LOC  = _pick_col(df_spk.columns, ["location", "place", "loc"])
COL_AGE  = _pick_col(df_spk.columns, ["age", "years"])
COL_GEN  = _pick_col(df_spk.columns, ["gen", "sex"])
COL_EDU  = _pick_col(df_spk.columns, ["education", "edu", "school"])

abbr2id, abbr2loc, abbr2age, abbr2gen, abbr2edu = {}, {}, {}, {}, {}
for _, r in df_spk.iterrows():
    abbr = r.get(COL_NAME, "").strip() if COL_NAME else ""
    if not abbr:
        continue
    if COL_ID:
        sid = r.get(COL_ID, "").strip()
        if sid:
            abbr2id[abbr] = sid
    if COL_LOC:
        loc = r.get(COL_LOC, "").strip()
        if loc:
            abbr2loc[abbr] = loc
    if COL_AGE:
        age = r.get(COL_AGE, "").strip()
        if age:
            abbr2age[abbr] = age
    if COL_GEN:
        gen = r.get(COL_GEN, "").strip()
        if gen:
            abbr2gen[abbr] = gen
    if COL_EDU:
        edu = r.get(COL_EDU, "").strip()
        if edu:
            abbr2edu[abbr] = edu

GEO = {}
if GEO_TSV is not None and Path(GEO_TSV).exists():
    df_geo = pd.read_csv(GEO_TSV, sep="\t", dtype=str)
    for _, r in df_geo.iterrows():
        place = str(r["place"]).strip()
        GEO[place] = (float(r["lat"]), float(r["lon"]))

def _spk_key(abbr: str) -> str:
    if abbr in abbr2id or abbr in abbr2loc or abbr in abbr2age or abbr in abbr2gen or abbr in abbr2edu:
        return abbr
    base = abbr.split("_")[0]
    return base

def speaker_id_from_abbr(abbr: str) -> str:
    if abbr.startswith("R"):
        return abbr
    k = _spk_key(abbr)
    return abbr2id.get(k, abbr)

def location_from_abbr(abbr: str) -> str:
    if abbr.startswith("R"):
        return ""
    k = _spk_key(abbr)
    return abbr2loc.get(k, "")

def speaker_age_from_abbr(abbr: str) -> str:
    k = _spk_key(abbr)
    return abbr2age.get(k, "_")

def speaker_gender_from_abbr(abbr: str) -> str:
    k = _spk_key(abbr)
    return abbr2gen.get(k, "_")

def speaker_edu_from_abbr(abbr: str) -> str:
    k = _spk_key(abbr)
    edu = abbr2edu.get(k, "").strip()
    return edu if edu else "no education"

def geo_from_abbr(abbr: str):
    loc = location_from_abbr(abbr)
    return None if not loc else GEO.get(loc)

print("✅ speaker entries:", len(abbr2id), "| age:", len(abbr2age), "| gender:", len(abbr2gen), "| edu:", len(abbr2edu))

✅ speaker entries: 135 | age: 79 | gender: 159 | edu: 24


In [ ]:
# =========================
# 5) LABEL MAPS
# =========================

def build_vocab(items, min_freq=1, specials=None):
    specials = specials or []
    c = Counter(items)
    vocab = {}
    for sp in specials:
        vocab[sp] = len(vocab)
    for k,v in c.most_common():
        if k in vocab:
            continue
        if v >= min_freq:
            vocab[k] = len(vocab)
    return vocab

def ensure_special(v: Dict[str,int], key: str):
    if key not in v:
        v[key] = len(v)
    return v

lemmas = [l for _,l,_,_,_ in train_ex]
upos   = [u for _,_,u,_,_ in train_ex]
feats  = [f for _,_,_,f,_ in train_ex]
xpos   = [x for _,_,_,_,x in train_ex]

lemma2id = build_vocab(lemmas, min_freq=2, specials=["<UNK>"])
upos2id  = build_vocab(upos,   min_freq=1, specials=["X","_"])
feat2id  = build_vocab(feats,  min_freq=1, specials=["_","<UNK>"])
xpos2id  = build_vocab(xpos,   min_freq=1, specials=["X","_","<UNK>"])

for sp in ["<UNK>"]:
    lemma2id = ensure_special(lemma2id, sp)
for sp in ["X","_"]:
    upos2id = ensure_special(upos2id, sp)
for sp in ["_","<UNK>"]:
    feat2id = ensure_special(feat2id, sp)
for sp in ["X","_","<UNK>"]:
    xpos2id = ensure_special(xpos2id, sp)

id2lemma = {i:s for s,i in lemma2id.items()}
id2upos  = {i:s for s,i in upos2id.items()}
id2feat  = {i:s for s,i in feat2id.items()}
id2xpos  = {i:s for s,i in xpos2id.items()}

print("✅ vocab sizes:",
      "lemma", len(lemma2id),
      "upos", len(upos2id),
      "feats", len(feat2id),
      "xpos", len(xpos2id))

✅ vocab sizes: lemma 2103 upos 16 feats 405 xpos 568


In [ ]:
# =========================
# 6) DATASET / MODEL
# =========================
from transformers import AutoConfig, AutoModel
from transformers import MT5EncoderModel  # <-- add this import
from transformers import T5EncoderModel, MT5EncoderModel, AutoConfig, AutoModel



class TokenDataset(Dataset):
    def __init__(self, examples, lemma2id, upos2id, feat2id, xpos2id):
        self.ex = examples
        self.lemma2id = lemma2id
        self.upos2id  = upos2id
        self.feat2id  = feat2id
        self.xpos2id  = xpos2id

    def __len__(self): return len(self.ex)

    def __getitem__(self, idx):
        form, lemma, upos, feats, xpos = self.ex[idx]
        lem_id = self.lemma2id.get(lemma, self.lemma2id["<UNK>"])
        up_id  = self.upos2id.get(upos,  self.upos2id["X"])
        fe_id  = self.feat2id.get(feats, self.feat2id.get("<UNK>", 0))
        xp_id  = self.xpos2id.get(xpos,  self.xpos2id.get("<UNK>", 0))
        return form, lem_id, up_id, fe_id, xp_id

@dataclass
class Batch:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    lem_y: torch.Tensor
    up_y: torch.Tensor
    fe_y: torch.Tensor
    xp_y: torch.Tensor
    tokens: List[str]

def make_collate(tokenizer, max_length: int):
    def collate(items):
        tokens = [it[0] for it in items]
        lem_y  = torch.tensor([it[1] for it in items], dtype=torch.long)
        up_y   = torch.tensor([it[2] for it in items], dtype=torch.long)
        fe_y   = torch.tensor([it[3] for it in items], dtype=torch.long)
        xp_y   = torch.tensor([it[4] for it in items], dtype=torch.long)

        enc = tokenizer(tokens, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        return Batch(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            lem_y=lem_y, up_y=up_y, fe_y=fe_y, xp_y=xp_y,
            tokens=tokens
        )
    return collate

def get_hidden_size_from_config(cfg):
    if hasattr(cfg, "hidden_size"):
        return int(cfg.hidden_size)
    if hasattr(cfg, "d_model"):
        return int(cfg.d_model)
    raise ValueError("Cannot infer hidden size from config.")

def mean_pool(last_hidden, attention_mask):
    # last_hidden: [B, T, H], mask: [B, T]
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed = (last_hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1.0)
    return summed / denom

class MultiHeadTokenTagger(nn.Module):
    def __init__(self, encoder_name: str, n_lemma: int, n_upos: int, n_feats: int, n_xpos: int, dropout: float = 0.1):
        super().__init__()
        self.encoder_name = encoder_name

        cfg = AutoConfig.from_pretrained(encoder_name)

        # ✅ mt5/t5 need encoder-only to avoid decoder_input_ids requirement
        mt = getattr(cfg, "model_type", "")

        if mt == "mt5":
            self.encoder = MT5EncoderModel.from_pretrained(encoder_name, torch_dtype=torch.float32)
            enc_cfg = self.encoder.config
        elif mt == "t5":
            # ✅ ByT5 uses T5-style config/model_type
            self.encoder = T5EncoderModel.from_pretrained(encoder_name, torch_dtype=torch.float32)
            enc_cfg = self.encoder.config
        else:
            self.encoder = AutoModel.from_pretrained(encoder_name, torch_dtype=torch.float32)
            enc_cfg = self.encoder.config

        h = get_hidden_size_from_config(enc_cfg)

        self.drop = nn.Dropout(dropout)
        self.lemma_head = nn.Linear(h, n_lemma)
        self.upos_head  = nn.Linear(h, n_upos)
        self.feat_head  = nn.Linear(h, n_feats)
        self.xpos_head  = nn.Linear(h, n_xpos)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        hidden = out.last_hidden_state  # [B, T, H]

        # ✅ pooling works for XLM-R / DeBERTa / mT5 consistently
        x = mean_pool(hidden, attention_mask)
        x = self.drop(x)

        return {
            "lemma_logits": self.lemma_head(x),
            "upos_logits":  self.upos_head(x),
            "feat_logits":  self.feat_head(x),
            "xpos_logits":  self.xpos_head(x),
        }

def ce_loss(logits, y):
    return nn.functional.cross_entropy(logits, y)

@torch.no_grad()
def evaluate(model, loader, amp_dtype=None):
    model.eval()
    total = 0
    loss_sum = 0.0
    corr_lem = corr_up = corr_fe = corr_xp = corr_full = 0

    for batch in loader:
        input_ids = batch.input_ids.to(device)
        attn      = batch.attention_mask.to(device)
        lem_y     = batch.lem_y.to(device)
        up_y      = batch.up_y.to(device)
        fe_y      = batch.fe_y.to(device)
        xp_y      = batch.xp_y.to(device)

        if amp_dtype is not None and device.type == "cuda":
            with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                out = model(input_ids, attn)
        else:
            out = model(input_ids, attn)

        loss = (
            ce_loss(out["lemma_logits"], lem_y) +
            ce_loss(out["upos_logits"],  up_y)  +
            ce_loss(out["feat_logits"],  fe_y)  +
            ce_loss(out["xpos_logits"],  xp_y)
        )

        loss_sum += float(loss.item()) * input_ids.size(0)
        total += input_ids.size(0)

        lem_p = out["lemma_logits"].argmax(dim=1)
        up_p  = out["upos_logits"].argmax(dim=1)
        fe_p  = out["feat_logits"].argmax(dim=1)
        xp_p  = out["xpos_logits"].argmax(dim=1)

        corr_lem += int((lem_p == lem_y).sum().item())
        corr_up  += int((up_p  == up_y).sum().item())
        corr_fe  += int((fe_p  == fe_y).sum().item())
        corr_xp  += int((xp_p  == xp_y).sum().item())
        corr_full += int(((lem_p==lem_y) & (up_p==up_y) & (fe_p==fe_y) & (xp_p==xp_y)).sum().item())

    return {
        "loss": loss_sum / max(1,total),
        "lemma_acc": corr_lem / max(1,total),
        "upos_acc":  corr_up  / max(1,total),
        "feats_acc": corr_fe  / max(1,total),
        "xpos_acc":  corr_xp  / max(1,total),
        "full_acc":  corr_full/ max(1,total),
        "n": total
    }

def save_run_artifacts(out_dir: Path, tokenizer):
    out_dir.mkdir(parents=True, exist_ok=True)
    tokenizer.save_pretrained(str(out_dir))
    json.dump(lemma2id, open(out_dir/"lemma2id.json","w",encoding="utf-8"), ensure_ascii=False, indent=2)
    json.dump(upos2id,  open(out_dir/"upos2id.json","w",encoding="utf-8"),  ensure_ascii=False, indent=2)
    json.dump(feat2id,  open(out_dir/"feature2id.json","w",encoding="utf-8"),ensure_ascii=False, indent=2)
    json.dump(xpos2id,  open(out_dir/"xpos2id.json","w",encoding="utf-8"),  ensure_ascii=False, indent=2)

print("✅ model/dataset ready")

✅ model/dataset ready


In [ ]:
# =========================
# 7) EXB PARSING + CoNLL-U EXPORT (with age/gender/education)
# =========================

GLOBAL_COLUMNS = "ID FORM LEMMA UPOS XPOS FEATS HEAD DEPREL DEPS MISC"

def parse_exb_utterances(exb_path: Path) -> List[Dict]:
    tree = etree.parse(str(exb_path))
    root = tree.getroot()

    timeline = root.find(".//common-timeline")
    tli_time: Dict[str, float] = {}
    for tli in timeline.iter("tli"):
        tid = tli.attrib.get("id")
        if tid and "time" in tli.attrib:
            try:
                tli_time[tid] = float(tli.attrib["time"])
            except:
                pass

    utterances: List[Dict] = []
    file_id = exb_path.stem

    for tier in root.iter("tier"):
        display = tier.attrib.get("display-name", "")
        if "_" not in display:
            continue
        speaker_abbr = display

        for ev in tier.iter("event"):
            if ev.text is None:
                continue
            start = ev.attrib.get("start")
            end   = ev.attrib.get("end")
            if start not in tli_time or end not in tli_time:
                continue
            utterances.append({
                "file_id": file_id,
                "speaker_abbr": speaker_abbr,
                "start_time": tli_time[start],
                "end_time": tli_time[end],
                "raw_text": ev.text
            })

    utterances.sort(key=lambda x: x["start_time"])
    return utterances

def write_conllu_sentence(
    f,
    sent_id: str,
    tokens: List[str],
    preds: List[Tuple[str,str,str,str]],  # (lemma, upos, xpos, feats)
    speaker_abbr: str,
    start_time: float,
    end_time: float
):
    spk_id = speaker_id_from_abbr(speaker_abbr)
    loc    = location_from_abbr(speaker_abbr)
    geo    = geo_from_abbr(speaker_abbr)

    age    = speaker_age_from_abbr(speaker_abbr)
    gender = speaker_gender_from_abbr(speaker_abbr)
    edu    = speaker_edu_from_abbr(speaker_abbr)

    f.write(f"# sent_id = {sent_id}\n")
    f.write(f"# text = {' '.join(tokens)}\n")
    f.write(f"# speaker = {spk_id}\n")
    f.write(f"# speaker_abbr = {speaker_abbr}\n")
    if loc:
        f.write(f"# location = {loc}\n")
    f.write(f"# speaker_age = {age}\n")
    f.write(f"# speaker_gender = {gender}\n")
    f.write(f"# speaker_education = {edu}\n")
    f.write(f"# start_time = {start_time:.3f}\n")
    f.write(f"# end_time = {end_time:.3f}\n")
    if geo is not None:
        lat, lon = geo
        f.write(f"# geo = {lat:.6f},{lon:.6f}\n")

    for i, (tok, (lem, up, xp, fe)) in enumerate(zip(tokens, preds), start=1):
        misc = f"MulText={xp}" if xp and xp != "_" else "_"
        f.write(f"{i}\t{tok}\t{lem}\t{up}\t{xp}\t{fe}\t_\t_\t_\t{misc}\n")
    f.write("\n")

def find_file_anywhere(root: Path, filename: str) -> Path:
    hits = [h for h in root.rglob(filename) if h.is_file()]
    if not hits:
        raise FileNotFoundError(f"Could not find {filename} under {root}")
    hits = sorted(hits, key=lambda p: len(str(p)))
    return hits[0]

def export_one_exb(exb_path: Path, out_path: Path, predict_batch_fn):
    utts = parse_exb_utterances(exb_path)
    sent_no = 0

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        f.write(f"# global.columns = {GLOBAL_COLUMNS}\n\n")
        for u in utts:
            tokens, special_mask = tokenize_with_rules(u["raw_text"])
            if not tokens:
                continue

            normal_tokens = [t for t, sp in zip(tokens, special_mask) if not sp]
            normal_preds = []
            bs = 256
            for i in range(0, len(normal_tokens), bs):
                normal_preds.extend(predict_batch_fn(normal_tokens[i:i+bs]))

            preds = []
            j = 0
            for tok, sp in zip(tokens, special_mask):
                if sp:
                    preds.append((tok, "X", "X", "_"))
                else:
                    preds.append(normal_preds[j]); j += 1

            sent_no += 1
            sent_id = f"{u['file_id']}-s{sent_no:04d}"
            write_conllu_sentence(
                f=f, sent_id=sent_id, tokens=tokens, preds=preds,
                speaker_abbr=u["speaker_abbr"],
                start_time=u["start_time"], end_time=u["end_time"]
            )

print("✅ EXB export ready")

✅ EXB export ready


In [ ]:
# =========================
# 8) TRAIN ONE MODEL + EXPORT TOR_C_0031.exb
# =========================

def train_one_model(
    model_name: str,
    out_dir: Path,
    epochs: int,
    bs: int,
    accum: int,
    lr: float,
    wd: float,
    max_length: int,
    use_fp16: bool,
    freeze_epochs: int = 1,
    warmup_ratio: float = 0.06,
    patience: int = 4,
    grad_clip: float = 1.0,
):
    print("\n==============================")
    print(f"🚀 Training: {model_name}")
    print(f"   epochs={epochs} bs={bs} accum={accum} eff_bs={bs*accum} lr={lr} wd={wd} max_len={max_length} fp16={use_fp16}")
    print(f"   out_dir={out_dir}")
    print("==============================")

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    save_run_artifacts(out_dir, tokenizer)

    train_ds = TokenDataset(train_ex, lemma2id, upos2id, feat2id, xpos2id)
    dev_ds   = TokenDataset(dev_ex,   lemma2id, upos2id, feat2id, xpos2id)
    test_ds  = TokenDataset(test_ex,  lemma2id, upos2id, feat2id, xpos2id)

    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2,
                              collate_fn=make_collate(tokenizer, max_length), pin_memory=True)
    dev_loader   = DataLoader(dev_ds,   batch_size=bs*2, shuffle=False, num_workers=2,
                              collate_fn=make_collate(tokenizer, max_length), pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=bs*2, shuffle=False, num_workers=2,
                              collate_fn=make_collate(tokenizer, max_length), pin_memory=True)

    model = MultiHeadTokenTagger(model_name, len(lemma2id), len(upos2id), len(feat2id), len(xpos2id), dropout=0.1).to(device)

    # Freeze encoder first
    for p in model.encoder.parameters():
        p.requires_grad = False

    optim = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=wd)

    total_steps = epochs * math.ceil(len(train_loader) / accum)
    warmup_steps = int(total_steps * warmup_ratio)
    sched = get_cosine_schedule_with_warmup(optim, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    amp_dtype = torch.float16 if (use_fp16 and device.type == "cuda") else None
    scaler = torch.amp.GradScaler("cuda") if amp_dtype is not None else None

    best_full = -1.0
    best_epoch = -1
    bad = 0

    def rebuild_optimizer_after_unfreeze():
        nonlocal optim, sched
        optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        total_steps2 = epochs * math.ceil(len(train_loader) / accum)
        warmup_steps2 = int(total_steps2 * warmup_ratio)
        sched = get_cosine_schedule_with_warmup(optim, num_warmup_steps=warmup_steps2, num_training_steps=total_steps2)

    log = []
    for epoch in range(1, epochs+1):
        model.train()
        t0 = time.time()
        loss_accum = 0.0
        seen = 0

        if epoch == freeze_epochs + 1:
            for p in model.encoder.parameters():
                p.requires_grad = True
            rebuild_optimizer_after_unfreeze()
            print("🔥 Encoder unfrozen")

        optim.zero_grad(set_to_none=True)

        for step, batch in enumerate(train_loader, start=1):
            input_ids = batch.input_ids.to(device, non_blocking=True)
            attn      = batch.attention_mask.to(device, non_blocking=True)
            lem_y     = batch.lem_y.to(device, non_blocking=True)
            up_y      = batch.up_y.to(device, non_blocking=True)
            fe_y      = batch.fe_y.to(device, non_blocking=True)
            xp_y      = batch.xp_y.to(device, non_blocking=True)

            if amp_dtype is not None:
                with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                    out = model(input_ids, attn)
                    loss = (
                        ce_loss(out["lemma_logits"], lem_y) +
                        ce_loss(out["upos_logits"],  up_y)  +
                        ce_loss(out["feat_logits"],  fe_y)  +
                        ce_loss(out["xpos_logits"],  xp_y)
                    ) / accum
                scaler.scale(loss).backward()
            else:
                out = model(input_ids, attn)
                loss = (
                    ce_loss(out["lemma_logits"], lem_y) +
                    ce_loss(out["upos_logits"],  up_y)  +
                    ce_loss(out["feat_logits"],  fe_y)  +
                    ce_loss(out["xpos_logits"],  xp_y)
                ) / accum
                loss.backward()

            loss_accum += float(loss.item()) * input_ids.size(0) * accum
            seen += input_ids.size(0)

            if (step % accum) == 0:
                if scaler is not None:
                    scaler.unscale_(optim)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(optim)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    optim.step()
                optim.zero_grad(set_to_none=True)
                sched.step()

        # save "last" checkpoint each epoch (overwrites)
        torch.save(model.state_dict(), out_dir/"last_epoch.pt")

        dev_metrics = evaluate(model, dev_loader, amp_dtype=amp_dtype)
        train_loss = loss_accum / max(1, seen)
        dt = time.time() - t0

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "dev_loss": dev_metrics["loss"],
            "dev_full": dev_metrics["full_acc"],
            "dev_lemma": dev_metrics["lemma_acc"],
            "dev_upos": dev_metrics["upos_acc"],
            "dev_feats": dev_metrics["feats_acc"],
            "dev_xpos": dev_metrics["xpos_acc"],
            "minutes": dt/60.0
        }
        log.append(row)

        print(f"[{epoch:02d}] train_loss={train_loss:.4f} dev_full={dev_metrics['full_acc']*100:.2f}% dev_loss={dev_metrics['loss']:.4f} ({dt/60:.1f}m)")

        if dev_metrics["full_acc"] > best_full + 1e-6:
            best_full = dev_metrics["full_acc"]
            best_epoch = epoch
            bad = 0
            torch.save(model.state_dict(), out_dir/"best_model.pt")
            json.dump({"best_dev_full": best_full, "best_epoch": best_epoch, "model_name": model_name},
                      open(out_dir/"best_meta.json","w",encoding="utf-8"), indent=2)
            print("💾 saved best_model.pt")
        else:
            bad += 1
            if bad >= patience:
                print(f"⏹️ Early stopping. best_dev_full={best_full*100:.2f}% at epoch {best_epoch}")
                break

        json.dump(log, open(out_dir/"train_log.json","w",encoding="utf-8"), indent=2)

    # ---- TEST best ----
    best = MultiHeadTokenTagger(model_name, len(lemma2id), len(upos2id), len(feat2id), len(xpos2id), dropout=0.1).to(device)
    best.load_state_dict(torch.load(out_dir/"best_model.pt", map_location=device))
    best_metrics = evaluate(best, test_loader, amp_dtype=amp_dtype)

    print(f"🧪 TEST full={best_metrics['full_acc']*100:.2f}% "
          f"lemma={best_metrics['lemma_acc']*100:.2f}% "
          f"upos={best_metrics['upos_acc']*100:.2f}% "
          f"feats={best_metrics['feats_acc']*100:.2f}% "
          f"xpos={best_metrics['xpos_acc']*100:.2f}%")

    return {
        "model_name": model_name,
        "out_dir": str(out_dir),
        "best_dev_full": best_full,
        "best_epoch": best_epoch,
        "test": best_metrics,
        "amp_dtype": ("fp16" if amp_dtype is not None else "fp32"),
        "max_length": max_length,
        "use_fp16": use_fp16
    }

@torch.no_grad()
def make_predict_batch_fn(model_dir: Path, model_name: str, max_length: int, use_fp16: bool):
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir), use_fast=True)

    lemma2id_ = json.load(open(model_dir/"lemma2id.json","r",encoding="utf-8"))
    upos2id_  = json.load(open(model_dir/"upos2id.json","r",encoding="utf-8"))
    feat2id_  = json.load(open(model_dir/"feature2id.json","r",encoding="utf-8"))
    xpos2id_  = json.load(open(model_dir/"xpos2id.json","r",encoding="utf-8"))

    id2lemma_ = {int(v):k for k,v in lemma2id_.items()}
    id2upos_  = {int(v):k for k,v in upos2id_.items()}
    id2feat_  = {int(v):k for k,v in feat2id_.items()}
    id2xpos_  = {int(v):k for k,v in xpos2id_.items()}

    m = MultiHeadTokenTagger(model_name, len(lemma2id_), len(upos2id_), len(feat2id_), len(xpos2id_), dropout=0.1).to(device)
    m.load_state_dict(torch.load(model_dir/"best_model.pt", map_location=device))
    m.eval()

    amp_dtype = torch.float16 if (use_fp16 and device.type == "cuda") else None

    def predict_batch(tokens: List[str]) -> List[Tuple[str,str,str,str]]:
        enc = tokenizer(tokens, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        input_ids = enc["input_ids"].to(device)
        attn      = enc["attention_mask"].to(device)

        if amp_dtype is not None:
            with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                out = m(input_ids, attn)
        else:
            out = m(input_ids, attn)

        lem = out["lemma_logits"].argmax(dim=1).tolist()
        up  = out["upos_logits"].argmax(dim=1).tolist()
        fe  = out["feat_logits"].argmax(dim=1).tolist()
        xp  = out["xpos_logits"].argmax(dim=1).tolist()

        preds = []
        for tok, li, ui, fi, xi in zip(tokens, lem, up, fe, xp):
            lem_s = id2lemma_.get(li, "<UNK>")
            up_s  = id2upos_.get(ui, "X")
            fe_s  = id2feat_.get(fi, "_")
            xp_s  = id2xpos_.get(xi, "X")

            if lem_s == "<UNK>":
                lem_s = tok
            if fe_s == "<UNK>":
                fe_s = "_"
            if xp_s == "<UNK>":
                xp_s = "X"
            preds.append((lem_s, up_s, xp_s, fe_s))
        return preds

    return predict_batch

print("✅ training + inference helpers ready")

✅ training + inference helpers ready


In [ ]:
# =========================
# 9) MULTI-RUN: MODELS + EXPORT TOR_C_0031.exb
# =========================

MODELS_TO_TRAIN = [
    # 1) Strong baseline (you already saw it win)
    {
        "name": "FacebookAI/xlm-roberta-large",
        "epochs": EPOCHS,
        "bs": 24,
        "accum": 4,
        "lr": 1e-5,
        "wd": WEIGHT_DECAY,
        "max_length": 16,
        "use_fp16": True,
    },

    # 2) Region-specialized (BCMS) - often very strong for morph/POS
    {
        "name": "classla/bcms-bertic",
        "epochs": EPOCHS,
        "bs": 64,
        "accum": 2,
        "lr": 2e-5,
        "wd": WEIGHT_DECAY,
        "max_length": 16,
        "use_fp16": True,
    },

    # 3) Big multilingual encoder (strong generalization)
    {
        "name": "google/rembert",
        "epochs": EPOCHS,
        "bs": 32,
        "accum": 2,          # eff_bs=64
        "lr": 2e-5,
        "wd": WEIGHT_DECAY,
        "max_length": 16,
        "use_fp16": True,
    },

    # 4) Byte-level T5 encoder (robust to spelling/noise/diacritics)
    # NOTE: requires MultiHeadTokenTagger to special-case model_type=="t5" -> T5EncoderModel
    {
        "name": "google/byt5-large",
        "epochs": EPOCHS,
        "bs": 8,
        "accum": 8,          # eff_bs=64 (keeps memory manageable)
        "lr": 1e-5,
        "wd": WEIGHT_DECAY,
        "max_length": 64,    # bytes need more length; adjust if OOM
        "use_fp16": False,   # start fp32 for stability; switch to True if stable
    },
]

# Find TOR_C_0031.exb
try:
    TOR_EXB = (EXB_DIR / "TOR_C_0031.exb") if (EXB_DIR / "TOR_C_0031.exb").exists() else None
    if TOR_EXB is None:
        TOR_EXB = find_file_anywhere(Path("/content/drive/MyDrive"), "TOR_C_0031.exb")
except Exception as e:
    TOR_EXB = None
    print("⚠️ Could not auto-find TOR_C_0031.exb:", e)

print("🎯 TOR_EXB:", TOR_EXB)

results = {}
for cfg in MODELS_TO_TRAIN:
    model_name = cfg["name"]
    safe = model_name.replace("/", "_")
    out_dir = MODELS_ROOT / safe

    res = train_one_model(
        model_name=model_name,
        out_dir=out_dir,
        epochs=cfg["epochs"],
        bs=cfg["bs"],
        accum=cfg["accum"],
        lr=cfg["lr"],
        wd=cfg["wd"],
        max_length=cfg["max_length"],
        use_fp16=cfg["use_fp16"],
        freeze_epochs=FREEZE_EPOCHS,
        patience=PATIENCE,
        warmup_ratio=WARMUP_RATIO,
        grad_clip=GRAD_CLIP,
    )
    results[model_name] = res

    # Export TOR_C_0031.exb using the best checkpoint
    if TOR_EXB is not None:
        predict_fn = make_predict_batch_fn(
            out_dir, model_name,
            max_length=cfg["max_length"],
            use_fp16=cfg["use_fp16"]
        )
        out_conllu = PRED_ROOT / safe / "TOR_C_0031.pred.conllu"
        export_one_exb(TOR_EXB, out_conllu, predict_fn)
        results[model_name]["TOR_C_0031_pred"] = str(out_conllu)
        print("📝 wrote:", out_conllu)

    # cleanup GPU between runs
    torch.cuda.empty_cache()

json.dump(results, open(MODELS_ROOT/"summary_results.json","w",encoding="utf-8"), indent=2, ensure_ascii=False)
print("\n✅ DONE. Summary saved to:", MODELS_ROOT/"summary_results.json")
print("✅ Predictions saved under:", PRED_ROOT)


🎯 TOR_EXB: /content/drive/MyDrive/TorlakData/exb_corrected/TOR_C_0031.exb

🚀 Training: FacebookAI/xlm-roberta-large
   epochs=30 bs=24 accum=4 eff_bs=96 lr=1e-05 wd=0.01 max_len=16 fp16=True
   out_dir=/content/drive/MyDrive/TorlakTag/models/multirun_20260213_064842/FacebookAI_xlm-roberta-large


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[01] train_loss=22.6551 dev_full=0.00% dev_loss=22.2628 (1.3m)
💾 saved best_model.pt
🔥 Encoder unfrozen
[02] train_loss=15.9828 dev_full=33.93% dev_loss=9.4095 (5.6m)
💾 saved best_model.pt
[03] train_loss=7.3923 dev_full=53.65% dev_loss=5.0689 (5.7m)
💾 saved best_model.pt
[04] train_loss=4.6602 dev_full=60.51% dev_loss=3.9541 (5.7m)
💾 saved best_model.pt
[05] train_loss=3.6879 dev_full=64.01% dev_loss=3.4151 (5.7m)
💾 saved best_model.pt
[06] train_loss=3.1614 dev_full=66.76% dev_loss=3.0631 (5.8m)
💾 saved best_model.pt
[07] train_loss=2.7921 dev_full=68.38% dev_loss=2.9556 (5.7m)
💾 saved best_model.pt
[08] train_loss=2.5109 dev_full=70.98% dev_loss=2.7304 (5.7m)
💾 saved best_model.pt
[09] train_loss=2.2900 dev_full=71.38% dev_loss=2.6293 (5.7m)
💾 saved best_model.pt
[10] train_loss=2.0991 dev_full=72.16% dev_loss=2.5252 (5.7m)
💾 saved best_model.pt
[11] train_loss=1.9516 dev_full=72.49% dev_loss=2.4766 (5.7m)
💾 saved best_model.pt
[12] train_loss=1.8252 dev_full=73.74% dev_loss=2.4340 

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/328 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
T5EncoderModel LOAD REPORT from: google/byt5-large
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

[01] train_loss=22.7292 dev_full=0.00% dev_loss=22.6344 (5.9m)
💾 saved best_model.pt
🔥 Encoder unfrozen


OutOfMemoryError: CUDA out of memory. Tried to allocate 24.00 MiB. GPU 0 has a total capacity of 22.03 GiB of which 7.12 MiB is free. Including non-PyTorch memory, this process has 22.02 GiB memory in use. Of the allocated memory 20.28 GiB is allocated by PyTorch, and 1.50 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
rembert_results = results["google/rembert"]
test_metrics = rembert_results["test"]

test_full = test_metrics['full_acc'] * 100
test_lemma = test_metrics['lemma_acc'] * 100
test_upos = test_metrics['upos_acc'] * 100
test_feats = test_metrics['feats_acc'] * 100
test_xpos = test_metrics['xpos_acc'] * 100

print(f"🧪 TEST full={test_full:.2f}% lemma={test_lemma:.2f}% upos={test_upos:.2f}% feats={test_feats:.2f}% xpos={test_xpos:.2f}%")

🧪 TEST full=79.14% lemma=91.00% upos=92.59% feats=86.01% xpos=83.39%


In [ ]:
model_name = rembert_results["model_name"]
model_dir = Path(rembert_results["out_dir"])
max_length = rembert_results["max_length"]
use_fp16 = rembert_results["use_fp16"]

predict_fn = make_predict_batch_fn(
    model_dir,
    model_name,
    max_length=max_length,
    use_fp16=use_fp16
)

safe_model_name = model_name.replace("/", "_")
out_conllu = PRED_ROOT / safe_model_name / "TOR_C_0031.pred.conllu"

# Ensure TOR_EXB is defined; it should be from the previous run
if 'TOR_EXB' not in locals() or TOR_EXB is None:
    try:
        TOR_EXB = find_file_anywhere(Path("/content/drive/MyDrive"), "TOR_C_0031.exb")
        print(f"🎯 Auto-found TOR_EXB: {TOR_EXB}")
    except Exception as e:
        print(f"⚠️ Could not find TOR_C_0031.exb: {e}")
        TOR_EXB = None

if TOR_EXB is not None:
    export_one_exb(TOR_EXB, out_conllu, predict_fn)
    print(f"📝 Predictions for TOR_C_0031.exb exported to: {out_conllu}")
else:
    print("Error: TOR_C_0031.exb file not found, cannot export predictions.")

Loading weights:   0%|          | 0/521 [00:00<?, ?it/s]

RemBertModel LOAD REPORT from: google/rembert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📝 Predictions for TOR_C_0031.exb exported to: /content/drive/MyDrive/TorlakData/conllu_preds_20260213_064842/google_rembert/TOR_C_0031.pred.conllu
